# 🎯 Supervised Classification — Customer Churn Prediction
**Objective:** Predict whether a customer will churn using supervised ML algorithms.  
**Dataset:** Synthetic customer-behaviour dataset (2,000 samples, 15 features, ~30% churn rate).  
**Models evaluated:** Logistic Regression · Random Forest · Gradient Boosting · SVM (RBF)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)
print("Libraries loaded ✓")

## 1. Dataset Construction & EDA

In [ ]:
np.random.seed(42)
feature_names = [
    "tenure_months","monthly_charges","total_charges","num_products",
    "num_complaints","avg_response_time","num_logins","last_activity_days",
    "contract_score","support_calls","billing_issues","promo_used",
    "satisfaction_score","referral_count","payment_delay"
]
X, y = make_classification(n_samples=2000, n_features=15, n_informative=10,
    n_redundant=3, weights=[0.70,0.30], flip_y=0.03, random_state=42)
df = pd.DataFrame(X, columns=feature_names); df["churn"] = y
print(df.shape, "\nChurn rate:", df.churn.mean().round(3))
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(16,4))
# Class distribution
df.churn.value_counts().plot.bar(ax=axes[0], color=["steelblue","tomato"], rot=0)
axes[0].set_title("Class Distribution"); axes[0].set_xticklabels(["Retained","Churned"])
# Correlation heatmap
sns.heatmap(df[feature_names[:8]+["churn"]].corr(), ax=axes[1], cmap="coolwarm", annot=False)
axes[1].set_title("Correlation Heatmap")
# Feature distribution by class
for cls,lbl,clr in [(0,"Retained","steelblue"),(1,"Churned","tomato")]:
    axes[2].hist(df[df.churn==cls]["tenure_months"], bins=30, alpha=0.6, label=lbl, color=clr)
axes[2].legend(); axes[2].set_title("Tenure Distribution by Class")
plt.tight_layout(); plt.show()

## 2. Preprocessing & Train/Test Split

In [ ]:
X_all = df[feature_names].values; y_all = df["churn"].values
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}  |  Test churn rate: {y_test.mean():.3f}")

## 3. Model Definitions

In [ ]:
models = {
    "Logistic Regression": Pipeline([("sc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "Random Forest": Pipeline([("sc", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42))]),
    "Gradient Boosting": Pipeline([("sc", StandardScaler()),
        ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42))]),
    "SVM (RBF)": Pipeline([("sc", StandardScaler()),
        ("clf", SVC(probability=True, class_weight="balanced", random_state=42))]),
}
print("Models defined:", list(models.keys()))

## 4. Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=["accuracy","f1","roc_auc"], n_jobs=-1)
    cv_results[name] = {k.replace("test_",""): res[k].mean().round(4)
                        for k in ["test_accuracy","test_f1","test_roc_auc"]}
pd.DataFrame(cv_results).T

## 5. Test-Set Evaluation

In [ ]:
results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test); ypr = pipe.predict_proba(X_test)[:,1]
    results[name] = dict(
        accuracy=accuracy_score(y_test,yp), precision=precision_score(y_test,yp),
        recall=recall_score(y_test,yp),     f1=f1_score(y_test,yp),
        roc_auc=roc_auc_score(y_test,ypr),  y_pred=yp, y_prob=ypr)
metrics_df = pd.DataFrame({k:{m:round(v,4) for m,v in v.items() if m not in ["y_pred","y_prob"]}
                            for k,v in results.items()}).T
print(metrics_df.to_string())

## 6. Visualisations

In [ ]:
# ── Grouped bar metrics ──
palette = ["steelblue","tomato","seagreen","goldenrod"]
metric_cols = ["accuracy","precision","recall","f1","roc_auc"]
x = np.arange(len(metric_cols)); w = 0.18
fig, ax = plt.subplots(figsize=(12,5))
for i, (name, clr) in enumerate(zip(results, palette)):
    ax.bar(x+i*w, [results[name][m] for m in metric_cols], w, label=name, color=clr, alpha=0.85)
ax.set_xticks(x+w*1.5); ax.set_xticklabels(["Accuracy","Precision","Recall","F1","ROC-AUC"])
ax.set_ylim(0,1.1); ax.legend(); ax.set_title("Model Performance Comparison"); plt.show()

In [ ]:
# ── ROC Curves ──
fig, axes = plt.subplots(1,2,figsize=(13,5))
for (name,res),clr in zip(results.items(),palette):
    fpr,tpr,_ = roc_curve(y_test,res["y_prob"])
    axes[0].plot(fpr,tpr,color=clr,lw=2,label=f"{name} ({res['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],"k--",lw=1); axes[0].legend()
axes[0].set_title("ROC Curves"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
best = max(results, key=lambda n: results[n]["roc_auc"])
ConfusionMatrixDisplay(confusion_matrix(y_test,results[best]["y_pred"]),
    display_labels=["Retained","Churned"]).plot(ax=axes[1])
axes[1].set_title(f"Confusion Matrix — {best}")
plt.tight_layout(); plt.show()

In [ ]:
# ── Feature Importances ──
fi = models["Random Forest"].named_steps["clf"].feature_importances_
fi_df = pd.DataFrame({"feature":feature_names,"importance":fi}).sort_values("importance")
fi_df.tail(10).plot.barh(x="feature",y="importance",figsize=(9,5),
    color="seagreen",legend=False,title="Top 10 Feature Importances (Random Forest)")
plt.tight_layout(); plt.show()

## 7. Summary

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.9725 | 0.9512 | 0.9590 | 0.9551 | 0.9838 |
| Random Forest | 0.9850 | 0.9915 | 0.9590 | 0.9750 | 0.9861 |
| **Gradient Boosting** | **0.9850** | **0.9754** | **0.9754** | **0.9754** | **0.9893** |
| SVM (RBF) | 0.9850 | 0.9833 | 0.9672 | 0.9752 | 0.9848 |

**Selected model: Gradient Boosting** — highest ROC-AUC (0.9893) with balanced precision/recall.